# 🌾 Rice Vision AI — Colab Inference Server

Notebook này vận hành dịch vụ suy luận Rice Vision AI trên Google Colab GPU (Tesla T4) với ngrok tunnel.

### Hướng dẫn vận hành nhanh:
1. **GPU Runtime**: Chọn menu **Runtime** → **Change runtime type** → **T4 GPU**.
2. **Mount Drive & Đường dẫn**: Chạy Cell 2 để mount Drive và trỏ `PROJECT_ROOT` tới thư mục `MAIN_SOURCES`.
3. **Cài đặt thư viện**: Chạy Cell 3 để cài đặt đúng gói theo `requirements-colab.txt`.
4. **Cấu hình & Model**: Khai báo token ngrok và đường dẫn model trong file `AI_SERVICES/.env`. Cell 4 nạp cấu hình tự động.
5. **Preflight**: Cell 5 nạp trước các model và kiểm tra tính hợp lệ trước khi khởi chạy mạng.
6. **Khởi chạy Server**: Cell 6 khởi chạy FastAPI server cục bộ, kiểm tra liveness và mở ngrok tunnel ra Internet.
7. **Dừng / Đổi model**: Khi cần đổi model, chạy Cell 8 để đóng tunnel và tắt server, sửa `.env` rồi chạy lại từ Cell 4.


In [16]:
# Cell 2: Mount Google Drive & Khai báo PROJECT_ROOT duy nhất
from google.colab import drive
import os
import sys
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Khai báo PROJECT_ROOT tới MAIN_SOURCES (Chỉnh sửa nếu vị trí thư mục của bạn khác)
PROJECT_ROOT = Path('/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES')

# 3. Kiểm tra tính hợp lệ của cây thư mục
AI_SERVICES_DIR = PROJECT_ROOT / 'AI_SERVICES'
SRC_DIR = AI_SERVICES_DIR / 'src'
RICE_AI_DIR = SRC_DIR / 'rice_ai'
ENV_FILE = AI_SERVICES_DIR / '.env'

if not RICE_AI_DIR.is_dir():
    raise FileNotFoundError(
        f"❌ Không tìm thấy package mã nguồn tại: {RICE_AI_DIR}\n"
        f"Vui lòng kiểm tra lại giá trị PROJECT_ROOT trên Google Drive!"
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(AI_SERVICES_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVICES_DIR))

print(f"✅ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✅ AI_SERVICES_DIR: {AI_SERVICES_DIR}")
print("✅ Source package 'rice_ai' đã sẵn sàng trong sys.path.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ PROJECT_ROOT: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES
✅ AI_SERVICES_DIR: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/AI_SERVICES
✅ Source package 'rice_ai' đã sẵn sàng trong sys.path.


In [17]:
# Cell 3: Cài đặt Dependencies từ requirements-colab.txt & Kiểm tra phần cứng
import sys
import subprocess
import torch

# 1. Cài đặt các gói phụ thuộc
req_colab = AI_SERVICES_DIR / 'requirements-colab.txt'
if req_colab.is_file():
    print(f"📦 Đang cài đặt thư viện từ {req_colab}...")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_colab)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"⚠️ Cảnh báo cài đặt: {res.stderr[:500]}")
    else:
        print("✅ Đã cài đặt hoàn tất các thư viện Colab.")
else:
    print(f"⚠️ Không tìm thấy {req_colab}. Tiếp tục với môi trường hiện có.")

# 2. Kiểm tra thông tin phần cứng và GPU
print("\n--- Môi trường tính toán ---")
print(f"Python: {sys.version.split()[0]}")
import sklearn
print(f"scikit-learn: {sklearn.__version__}")
import numpy as np
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

cuda_avail = torch.cuda.is_available()
print(f"CUDA Available: {cuda_avail}")
if cuda_avail:
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU Device: {device_name}")
else:
    print("⚠️ CẢNH BÁO: Không có GPU! Vào menu Runtime -> Change runtime type -> T4 GPU để tăng tốc.")


📦 Đang cài đặt thư viện từ /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/AI_SERVICES/requirements-colab.txt...
✅ Đã cài đặt hoàn tất các thư viện Colab.

--- Môi trường tính toán ---
Python: 3.13.15
scikit-learn: 1.6.1
NumPy: 2.1.3
PyTorch: 2.11.0+cu128
CUDA Available: True
GPU Device: Tesla T4


In [18]:
# Cell 4: Nạp Settings và cấu hình notebook từ .env
import os
from dotenv import dotenv_values
from rice_ai.settings import Settings

# Settings quản lý đường dẫn model, thiết bị YOLO, port và concurrency.
settings = Settings(env_file=ENV_FILE)

# Ngrok là cấu hình vận hành của notebook, không thuộc Settings của inference runtime.
file_env = dotenv_values(ENV_FILE) if ENV_FILE.is_file() else {}
def notebook_env_value(key: str, default: str = "") -> str:
    value = os.environ[key] if key in os.environ else file_env.get(key, default)
    return str(value or default).strip()

server_host = "127.0.0.1"
server_port = settings.port_ai
ngrok_auth_token = notebook_env_value("NGROK_AUTH_TOKEN")
ngrok_domain = notebook_env_value("NGROK_DOMAIN")

# Thông báo thiết lập cốt lõi; không in token bí mật.
print("--- Cấu hình Hệ thống (Settings) ---")
print(f"Host / Port      : {server_host} / {server_port}")
print(f"YOLO Device      : {settings.yolo_device_str} (Giải quyết: {settings.get_yolo_device()})")
print(f"YOLO Model Path  : {settings.get_yolo_path()}")
print(f"CNN Model Path   : {settings.get_cnn_path()}")
print(f"Regression Dir   : {settings.get_regression_dir()}")
print(f"Max Concurrency  : {settings.max_concurrent_inferences}")
print(f"Ngrok Auth Token : {'Đã cấu hình (Bảo mật)' if ngrok_auth_token else 'CHƯA CẤU HÌNH ⚠️'}")
print(f"Ngrok Domain     : {ngrok_domain or '(Sinh ngẫu nhiên)'}")


--- Cấu hình Hệ thống (Settings) ---
Host / Port      : 127.0.0.1 / 8000
YOLO Device      : auto (Giải quyết: cuda:0)
YOLO Model Path  : /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/all-new-data-v1.yolov8_yolov8s-seg_trained/weights/best.pt
CNN Model Path   : /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras
Regression Dir   : /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/LINEAR_REGRESSION_MODEL/models
Max Concurrency  : 1
Ngrok Auth Token : Đã cấu hình (Bảo mật)
Ngrok Domain     : provolone-duress-probably.ngrok-free.dev


In [19]:
# Cell 5: Preflight Verification — Kiểm tra tính toàn vẹn của mô hình trước khi mở mạng
from rice_ai.models.vision_models import VisionModelProvider
from rice_ai.models.regression_loader import LoadedRegressionProvider

print("🔍 Đang tiến hành Preflight Verification...")

# 1. Khởi tạo và nạp kiểm tra Vision Models (YOLO & CNN)
vision_prov = VisionModelProvider(settings)
try:
    yolo_model = vision_prov.get_yolo_model()
    print(f"✅ YOLO Model đã nạp thành công trên thiết bị: {settings.get_yolo_device()}")
except Exception as e:
    raise RuntimeError(f"❌ Nạp YOLO Model thất bại: {e}")

try:
    cnn_model = vision_prov.get_cnn_model()
    print("✅ CNN Classifier (DenseNet121) đã nạp thành công.")
except Exception as e:
    raise RuntimeError(f"❌ Nạp CNN Model thất bại: {e}")

# 2. Khởi tạo và nạp kiểm tra Regression Bundle
reg_prov = LoadedRegressionProvider(settings)
try:
    loaded_reg = reg_prov.get_regression()
    bundle_layout = "legacy adapter" if loaded_reg.is_legacy else "canonical folder"
    scaler_name = type(loaded_reg.scaler).__name__ if loaded_reg.scaler is not None else "None (preprocessing=none)"
    print(f"✅ Regression Model đã nạp thành công: {loaded_reg.model_name}")
    print(f"   - Model Class    : {type(loaded_reg.model).__name__}")
    print(f"   - Schema Version : {loaded_reg.schema_version}")
    print(f"   - Bundle Layout  : {bundle_layout}")
    print(f"   - Scaler         : {scaler_name}")
    print(f"   - Bundle Dir     : {loaded_reg.model_dir}")
    for warning in loaded_reg.warnings:
        print(f"   - Warning        : {warning}")
except Exception as e:
    raise RuntimeError(f"❌ Nạp Regression Model thất bại: {e}") from e

print("\n🎉 Tất cả mô hình đã vượt qua Preflight! Sẵn sàng khởi động Server.")


🔍 Đang tiến hành Preflight Verification...
✅ YOLO Model đã nạp thành công trên thiết bị: cuda:0
📦 Đang nạp mô hình CNN từ: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras (Keras 3)...
✅ Nạp mô hình CNN thành công!
✅ CNN Classifier (DenseNet121) đã nạp thành công.
✅ Regression Model đã nạp thành công: ExtraTrees
   - Model Class    : ExtraTreesRegressor
   - Schema Version : 31v1
   - Bundle Layout  : legacy adapter
   - Scaler         : StandardScaler
   - Bundle Dir     : /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/LINEAR_REGRESSION_MODEL/models
   - Warning        : LEGACY_LAYOUT: Đang nạp mô hình qua adapter tương thích legacy (best_tree_ensemble_model.joblib).

🎉 Tất cả mô hình đã vượt qua Preflight! Sẵn sàng khởi động Server.


In [20]:
# Cell 6: Khởi động FastAPI Server và thiết lập ngrok Tunnel
import asyncio
import time
import httpx
from pyngrok import ngrok
import uvicorn
import nest_asyncio
nest_asyncio.apply()

from rice_ai.api.application import create_app

# 1. Kiểm tra trạng thái chạy trước đó để tránh conflict cổng
if 'server_task' in globals() and server_task is not None and not server_task.done():
    print("⚠️ Server đang chạy ở background! Vui lòng chạy Cell 8 để Stop trước khi Start lại.")
else:
    # 2. Tạo ứng dụng FastAPI với providers đã preflight
    app = create_app(settings, vision_provider=vision_prov, regression_provider=reg_prov)

    # 3. Khởi chạy Uvicorn Server trong background task
    config = uvicorn.Config(app=app, host="127.0.0.1", port=server_port, log_level="info", loop="asyncio")
    server = uvicorn.Server(config)
    loop = asyncio.get_event_loop()
    server_task = loop.create_task(server.serve())

    # 4. Kiểm tra sức khỏe Server cục bộ trước khi mở Tunnel
    print("⏳ Đang đợi server khởi động...")
    local_ready = False
    for attempt in range(20):
        await asyncio.sleep(0.5)
        try:
            async with httpx.AsyncClient() as client:
                resp = await client.get(f"http://127.0.0.1:{server_port}/health", timeout=1.0)
                if resp.status_code == 200:
                    local_ready = True
                    break
        except Exception:
            pass

    if not local_ready:
        server.should_exit = True
        raise RuntimeError("❌ Server không phản hồi /health tại localhost sau 10 giây!")

    print(f"✅ Server nội bộ đã phản hồi 200 OK tại port {server_port}!")

    # 5. Mở ngrok Tunnel
    if not ngrok_auth_token:
        raise ValueError("❌ Thiếu NGROK_AUTH_TOKEN trong .env! Vui lòng điền token vào .env.")

    ngrok.set_auth_token(ngrok_auth_token)
    domain = ngrok_domain.replace("https://", "").replace("http://", "") if ngrok_domain else None

    if domain:
        print(f"🌐 Đang mở tunnel với Static Domain: {domain}...")
        current_tunnel = ngrok.connect(server_port, domain=domain)
    else:
        print("🌐 Đang mở tunnel với Dynamic Domain...")
        current_tunnel = ngrok.connect(server_port)

    public_url = current_tunnel.public_url
    print("\n" + "=" * 62)
    print(f"  🚀 NGROK PUBLIC URL: {public_url}")
    print("  Ứng dụng web/di động sẽ sử dụng URL này để gọi API suy luận.")
    print("=" * 62 + "\n")

    # 6. Đồng bộ URL mới vào .env (không làm mất comment hay các key khác)
    try:
        if ENV_FILE.is_file():
            lines = ENV_FILE.read_text(encoding="utf-8").splitlines(keepends=True)
            new_lines = []
            updated = False
            for line in lines:
                if line.strip().startswith("AI_SERVER_URL="):
                    new_lines.append(f"AI_SERVER_URL={public_url}\n")
                    updated = True
                else:
                    new_lines.append(line)
            if not updated:
                new_lines.append(f"\nAI_SERVER_URL={public_url}\n")
            ENV_FILE.write_text("".join(new_lines), encoding="utf-8")
            print("💾 Đã tự động đồng bộ AI_SERVER_URL vào file .env!")
    except Exception as ex:
        print(f"⚠️ Cập nhật .env không thành công: {ex}")


INFO:     Started server process [23273]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


⏳ Đang đợi server khởi động...
INFO:     127.0.0.1:57936 - "GET /health HTTP/1.1" 200 OK
✅ Server nội bộ đã phản hồi 200 OK tại port 8000!
🌐 Đang mở tunnel với Static Domain: provolone-duress-probably.ngrok-free.dev...

  🚀 NGROK PUBLIC URL: https://provolone-duress-probably.ngrok-free.dev
  Ứng dụng web/di động sẽ sử dụng URL này để gọi API suy luận.

💾 Đã tự động đồng bộ AI_SERVER_URL vào file .env!


In [21]:
# Cell 7: Smoke Test & Kiểm tra E2E Tùy chọn
import httpx

# Cờ chạy kiểm thử toàn diện E2E (Đặt True nếu muốn gửi ảnh thật để đo độ trễ và số hạt)
RUN_E2E = False

print("🔎 Đang kiểm tra Public URL qua Internet...")
async with httpx.AsyncClient() as client:
    # 1. Health check
    h_res = await client.get(f"{public_url}/health", timeout=10.0)
    print(f"GET /health: {h_res.status_code} -> {h_res.json()}")

    # 2. System Status check
    s_res = await client.get(f"{public_url}/api/status", timeout=10.0)
    print(f"GET /api/status: {s_res.status_code} -> readiness={s_res.json().get('readiness')}")
    print(f"Components: {s_res.json().get('components')}")

    # 3. E2E Predict (Nếu bật)
    if RUN_E2E:
        test_img_path = PROJECT_ROOT / "RESULTS" / "M001A_top_sample.jpg"
        if not test_img_path.is_file():
            test_img_path = AI_SERVICES_DIR / "tests" / "fixtures" / "sample.jpg"

        if test_img_path.is_file():
            print(f"\n🧪 Đang chạy E2E Predict với ảnh: {test_img_path.name}...")
            with open(test_img_path, "rb") as f:
                form_data = {
                    "diam": "1.78",
                    "height": "3.39",
                    "empty": "1.09",
                    "wall_thickness": "0.1",
                    "estimator_mode": "auto",
                    "debug": "true",
                }
                files = {"file": (test_img_path.name, f, "image/jpeg")}
                p_res = await client.post(f"{public_url}/predict", data=form_data, files=files, timeout=180.0)
                print(f"POST /predict: {p_res.status_code}")
                if p_res.status_code == 200:
                    data = p_res.json()
                    est = data.get("estimation", {})
                    timings = data.get("timings_ms", {})
                    print(f"🎯 Kết quả dự đoán: Final={est.get('final')} | Regression={est.get('regression_est')} | Method={est.get('method_used')}")
                    print(f"⏱️ Thời gian xử lý: Total={timings.get('total_ms')}ms | YOLO={timings.get('segmentation_ms')}ms | CNN={timings.get('classification_ms')}ms")
                else:
                    print(f"❌ Predict thất bại: {p_res.text}")
        else:
            print(f"⚠️ Không tìm thấy file ảnh mẫu để chạy E2E: {test_img_path}")


🔎 Đang kiểm tra Public URL qua Internet...
INFO:     136.85.133.29:0 - "GET /health HTTP/1.1" 200 OK
GET /health: 200 -> {'status': 'ok'}
INFO:     136.85.133.29:0 - "GET /api/status HTTP/1.1" 200 OK
GET /api/status: 200 -> readiness=ready
Components: {'yolo': {'status': 'loaded', 'file': 'best.pt'}, 'cnn': {'status': 'loaded', 'file': 'best_v3_step2.keras'}, 'regression': {'status': 'loaded', 'file': 'models'}}


In [22]:
# Cell 8: Dừng Server & Đóng Tunnel An toàn
import asyncio
from pyngrok import ngrok

print("🛑 Đang đóng ngrok tunnel và tắt server...")

# 1. Đóng tunnel hiện tại
if 'current_tunnel' in globals() and current_tunnel is not None:
    try:
        ngrok.disconnect(current_tunnel.public_url)
        print(f"✅ Đã đóng ngrok tunnel: {current_tunnel.public_url}")
    except Exception as ex:
        print(f"⚠️ Đóng tunnel lỗi hoặc tunnel đã đóng: {ex}")
    current_tunnel = None

# 2. Tắt server Uvicorn
if 'server' in globals() and server is not None:
    server.should_exit = True
    print("⏳ Đã gửi tín hiệu should_exit cho Uvicorn server.")

if 'server_task' in globals() and server_task is not None:
    try:
        await asyncio.wait_for(server_task, timeout=5.0)
        print("✅ Server task đã kết thúc an toàn.")
    except asyncio.TimeoutError:
        print("⚠️ Server task timeout khi chờ kết thúc.")
    except Exception as ex:
        print(f"Server task kết thúc: {ex}")
    server_task = None

server = None
print("\n🎉 Hệ thống đã dừng hoàn toàn!")
print("💡 Để đổi model: Sửa file .env rồi chạy lại từ Cell 4.")
print("💡 Để nạp mã nguồn Python mới: Vào menu Runtime -> Restart session rồi chạy lại từ Cell 2.")


INFO:     Shutting down
INFO:     Waiting for background tasks to complete. (CTRL+C to force quit)


🛑 Đang đóng ngrok tunnel và tắt server...
✅ Đã đóng ngrok tunnel: https://provolone-duress-probably.ngrok-free.dev
⏳ Đã gửi tín hiệu should_exit cho Uvicorn server.
⚠️ Server task timeout khi chờ kết thúc.

🎉 Hệ thống đã dừng hoàn toàn!
💡 Để đổi model: Sửa file .env rồi chạy lại từ Cell 4.
💡 Để nạp mã nguồn Python mới: Vào menu Runtime -> Restart session rồi chạy lại từ Cell 2.
